In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob as gl
import scipy.optimize
from tqdm.auto import tqdm
from scipy.interpolate import InterpolatedUnivariateSpline
from scipy.spatial import distance
import pymap3d as pm
import pymap3d.vincenty as pmv

# Constants
CLIGHT = 299_792_458   # speed of light (m/s)
RE_WGS84 = 6_378_137   # earth semimajor axis (WGS84) (m)
OMGE = 7.2921151467E-5  # earth angular velocity (IS-GPS) (rad/s)

In [2]:
parent_folder = "../train"

# retrieve device.imu files from train
gnss_files = gl.glob(os.path.join(parent_folder, "*", "*", "device_gnss.csv"))
print(gnss_files) 

['../train/2021-01-04-US-SFO-2/GooglePixel5/device_gnss.csv', '../train/2021-01-04-US-SFO-2/XiaomiMi8/device_gnss.csv', '../train/2021-01-04-US-SFO-2/GooglePixel4/device_gnss.csv', '../train/2021-01-04-US-SFO-2/GooglePixel4XL/device_gnss.csv', '../train/2020-05-21-US-MTV-1/GooglePixel4/device_gnss.csv', '../train/2020-05-21-US-MTV-1/GooglePixel4XL/device_gnss.csv', '../train/2021-04-21-US-MTV-2/XiaomiMi8/device_gnss.csv', '../train/2021-04-21-US-MTV-2/SamsungGalaxyS20Ultra/device_gnss.csv', '../train/2020-11-23-US-MTV-1/XiaomiMi8/device_gnss.csv', '../train/2020-06-18-US-MTV-1/GooglePixel4/device_gnss.csv', '../train/2020-06-18-US-MTV-1/GooglePixel4XL/device_gnss.csv', '../train/2020-07-24-US-MTV-1/GooglePixel5/device_gnss.csv', '../train/2020-07-24-US-MTV-1/GooglePixel4/device_gnss.csv', '../train/2020-07-24-US-MTV-1/GooglePixel4XL/device_gnss.csv', '../train/2020-05-15-US-MTV-1/GooglePixel4XL/device_gnss.csv', '../train/2020-06-05-US-MTV-1/GooglePixel4/device_gnss.csv', '../train/202

In [3]:
gnss_list = []

for file in gnss_files:
    gnss_file = pd.read_csv(file)
    parts = file.split(os.sep) # eg ['train', '2021-12-28-US-MTV-1', 'GooglePixel5', 'device_imu.csv']
    gnss_file["drive_id"] = parts[-3] 
    gnss_file["device"] = parts[-2]
    gnss_list.append(gnss_file)

# combine all
gnss_df = pd.concat(gnss_list, ignore_index=True)
print(gnss_df.head())
print()
print(gnss_df["device"].unique())
print()
print(gnss_df["drive_id"].unique())

/var/folders/w8/d9x92py92j792gw_1kyfc8sh0000gn/T/ipykernel_34178/1769875542.py:4: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  gnss_file = pd.read_csv(file)


  MessageType  utcTimeMillis       TimeNanos  LeapSecond        FullBiasNanos  \
0         Raw  1609800004433  10686613000000        18.0 -1293824535820307301   
1         Raw  1609800004433  10686613000000        18.0 -1293824535820307301   
2         Raw  1609800004433  10686613000000        18.0 -1293824535820307301   
3         Raw  1609800004433  10686613000000        18.0 -1293824535820307301   
4         Raw  1609800004433  10686613000000        18.0 -1293824535820307301   

   BiasNanos  BiasUncertaintyNanos  DriftNanosPerSecond  \
0  -0.133871             25.289228            -9.770029   
1  -0.133871             25.289228            -9.770029   
2  -0.133871             25.289228            -9.770029   
3  -0.133871             25.289228            -9.770029   
4  -0.133871             25.289228            -9.770029   

   DriftUncertaintyNanosPerSecond  HardwareClockDiscontinuityCount  ...  \
0                         9.47592                               50  ...   
1       

In [4]:
# Satellite selection using carrier frequency error, elevation angle, and C/N0
def satellite_selection(df, column):
    """
    Args:
        df : DataFrame from device_gnss.csv
        column : Column name
    Returns:
        df: DataFrame with eliminated satellite signals
    """
    idx = df[column].notnull()
    idx &= df['CarrierErrorHz'] < 2.0e6  # carrier frequency error (Hz)
    idx &= df['SvElevationDegrees'] > 10.0  # elevation angle (deg)
    idx &= df['Cn0DbHz'] > 15.0  # C/N0 (dB-Hz)
    idx &= df['MultipathIndicator'] == 0 # Multipath flag

    return df[idx]


In [5]:
# Compute line-of-sight vector from user to satellite
def los_vector(xusr, xsat):
    """
    Args:
        xusr : user position in ECEF (m)
        xsat : satellite position in ECEF (m)
    Returns:
        u: unit line-of-sight vector in ECEF (m)
        rng: distance between user and satellite (m)
    """
    u = xsat - xusr
    rng = np.linalg.norm(u, axis=1).reshape(-1, 1)
    u /= rng
    
    return u, rng.reshape(-1)


# Compute Jacobian matrix
def jac_pr_residuals(x, xsat, pr, W):
    """
    Args:
        x : current position in ECEF (m)
        xsat : satellite position in ECEF (m)
        pr : pseudorange (m)
        W : weight matrix
    Returns:
        W*J : Jacobian matrix
    """
    u, _ = los_vector(x[:3], xsat)
    J = np.hstack([-u, np.ones([len(pr), 1])])  # J = [-ux -uy -uz 1]

    return W @ J


# Compute pseudorange residuals
def pr_residuals(x, xsat, pr, W):
    """
    Args:
        x : current position in ECEF (m)
        xsat : satellite position in ECEF (m)
        pr : pseudorange (m)
        W : weight matrix
    Returns:
        residuals*W : pseudorange residuals
    """
    u, rng = los_vector(x[:3], xsat)

    # Approximate correction of the earth rotation (Sagnac effect) often used in GNSS positioning
    rng += OMGE * (xsat[:, 0] * x[1] - xsat[:, 1] * x[0]) / CLIGHT

    # Add GPS L1 clock offset
    residuals = rng - (pr - x[3])

    return residuals @ W


# Compute Jacobian matrix
def jac_prr_residuals(v, vsat, prr, x, xsat, W):
    """
    Args:
        v : current velocity in ECEF (m/s)
        vsat : satellite velocity in ECEF (m/s)
        prr : pseudorange rate (m/s)
        x : current position in ECEF (m)
        xsat : satellite position in ECEF (m)
        W : weight matrix
    Returns:
        W*J : Jacobian matrix
    """
    u, _ = los_vector(x[:3], xsat)
    J = np.hstack([-u, np.ones([len(prr), 1])])

    return W @ J


# Compute pseudorange rate residuals
def prr_residuals(v, vsat, prr, x, xsat, W):
    """
    Args:
        v : current velocity in ECEF (m/s)
        vsat : satellite velocity in ECEF (m/s)
        prr : pseudorange rate (m/s)
        x : current position in ECEF (m)
        xsat : satellite position in ECEF (m)
        W : weight matrix
    Returns:
        residuals*W : pseudorange rate residuals
    """
    u, rng = los_vector(x[:3], xsat)
    rate = np.sum((vsat-v[:3])*u, axis=1) \
          + OMGE / CLIGHT * (vsat[:, 1] * x[0] + xsat[:, 1] * v[0]
                           - vsat[:, 0] * x[1] - xsat[:, 0] * v[1])

    residuals = rate - (prr - v[3])

    return residuals @ W

In [6]:
# Carrier smoothing of pseudarange
def carrier_smoothing(gnss_df):
    """
    Args:
        df : DataFrame from device_gnss.csv
    Returns:
        df: DataFrame with carrier-smoothing pseudorange 'pr_smooth'
    """
    carr_th = 1.5 # carrier phase jump threshold [m] ** 2.0 -> 1.5 **
    pr_th =  20.0 # pseudorange jump threshold [m]

    prsmooth = np.full_like(gnss_df['RawPseudorangeMeters'], np.nan)
    # Loop for each signal
    for (i, (svid_sigtype, df)) in enumerate((gnss_df.groupby(['Svid', 'SignalType']))):
        df = df.replace(
            {'AccumulatedDeltaRangeMeters': {0: np.nan}})  # 0 to NaN

        # Compare time difference between pseudorange/carrier with Doppler
        drng1 = df['AccumulatedDeltaRangeMeters'].diff() - df['PseudorangeRateMetersPerSecond']
        drng2 = df['RawPseudorangeMeters'].diff() - df['PseudorangeRateMetersPerSecond']

        # Check cycle-slip
        slip1 = (df['AccumulatedDeltaRangeState'].to_numpy() & 2**1) != 0  # reset flag
        slip2 = (df['AccumulatedDeltaRangeState'].to_numpy() & 2**2) != 0  # cycle-slip flag
        slip3 = np.fabs(drng1.to_numpy()) > carr_th # Carrier phase jump
        slip4 = np.fabs(drng2.to_numpy()) > pr_th # Pseudorange jump

        idx_slip = slip1 | slip2 | slip3 | slip4
        idx_slip[0] = True

        # groups with continuous carrier phase tracking
        df['group_slip'] = np.cumsum(idx_slip)

        # Psudorange - carrier phase
        df['dpc'] = df['RawPseudorangeMeters'] - df['AccumulatedDeltaRangeMeters']

        # Absolute distance bias of carrier phase
        meandpc = df.groupby('group_slip')['dpc'].mean()
        df = df.merge(meandpc, on='group_slip', suffixes=('', '_Mean'))

        # Index of original gnss_df
        idx = (gnss_df['Svid'] == svid_sigtype[0]) & (
        gnss_df['SignalType'] == svid_sigtype[1])

        # Carrier phase + bias
        prsmooth[idx] = df['AccumulatedDeltaRangeMeters'] + df['dpc_Mean']

    # If carrier smoothing is not possible, use original pseudorange
    idx_nan = np.isnan(prsmooth)
    prsmooth[idx_nan] = gnss_df['RawPseudorangeMeters'][idx_nan]
    gnss_df['pr_smooth'] = prsmooth

    return gnss_df

In [7]:
# Compute distance by Vincenty's formulae
def vincenty_distance(llh1, llh2):
    """
    Args:
        llh1 : [latitude,longitude] (deg)
        llh2 : [latitude,longitude] (deg)
    Returns:
        d : distance between llh1 and llh2 (m)
    """
    d, az = np.array(pmv.vdist(llh1[:, 0], llh1[:, 1], llh2[:, 0], llh2[:, 1]))

    return d


# Compute score
def calc_score(llh, llh_gt):
    """
    Args:
        llh : [latitude,longitude] (deg)
        llh_gt : [latitude,longitude] (deg)
    Returns:
        score : (m)
    """
    d = vincenty_distance(llh, llh_gt)
    score = np.mean([np.quantile(d, 0.50), np.quantile(d, 0.95)])

    return score

In [8]:
# GNSS single point positioning using pseudorange
def point_positioning(gnss_df):
    # Add nominal frequency to each signal
    # Note: GLONASS is an FDMA signal, so each satellite has a different frequency
    CarrierFrequencyHzRef = gnss_df.groupby(['Svid', 'SignalType'])[
        'CarrierFrequencyHz'].median()
    gnss_df = gnss_df.merge(CarrierFrequencyHzRef, how='left', on=[
                            'Svid', 'SignalType'], suffixes=('', 'Ref'))
    gnss_df['CarrierErrorHz'] = np.abs(
        (gnss_df['CarrierFrequencyHz'] - gnss_df['CarrierFrequencyHzRef']))

    # Carrier smoothing
    gnss_df = carrier_smoothing(gnss_df)

    # GNSS single point positioning
    utcTimeMillis = gnss_df['utcTimeMillis'].unique()
    nepoch = len(utcTimeMillis)
    x0 = np.zeros(4)  # [x,y,z,tGPSL1]
    v0 = np.zeros(4)  # [vx,vy,vz,dtGPSL1]
    x_wls = np.full([nepoch, 3], np.nan)  # For saving position
    v_wls = np.full([nepoch, 3], np.nan)  # For saving velocity
    cov_x = np.full([nepoch, 3, 3], np.nan) # For saving position covariance
    cov_v = np.full([nepoch, 3, 3], np.nan) # For saving velocity covariance

    # Loop for epochs
    for i, (t_utc, df) in enumerate(tqdm(gnss_df.groupby('utcTimeMillis'), total=nepoch)):
        # Valid satellite selection
        df_pr = satellite_selection(df, 'pr_smooth')
        df_prr = satellite_selection(df, 'PseudorangeRateMetersPerSecond')

        # Corrected pseudorange/pseudorange rate
        pr = (df_pr['pr_smooth'] + df_pr['SvClockBiasMeters'] - df_pr['IsrbMeters'] -
              df_pr['IonosphericDelayMeters'] - df_pr['TroposphericDelayMeters']).to_numpy()
        prr = (df_prr['PseudorangeRateMetersPerSecond'] +
               df_prr['SvClockDriftMetersPerSecond']).to_numpy()

        # Satellite position/velocity
        xsat_pr = df_pr[['SvPositionXEcefMeters', 'SvPositionYEcefMeters',
                         'SvPositionZEcefMeters']].to_numpy()
        xsat_prr = df_prr[['SvPositionXEcefMeters', 'SvPositionYEcefMeters',
                           'SvPositionZEcefMeters']].to_numpy()
        vsat = df_prr[['SvVelocityXEcefMetersPerSecond', 'SvVelocityYEcefMetersPerSecond',
                       'SvVelocityZEcefMetersPerSecond']].to_numpy()

        # Weight matrix for peseudorange/pseudorange rate
        Wx = np.diag(1 / df_pr['RawPseudorangeUncertaintyMeters'].to_numpy())
        Wv = np.diag(1 / df_prr['PseudorangeRateUncertaintyMetersPerSecond'].to_numpy())

        # Robust WLS requires accurate initial values for convergence,
        # so perform normal WLS for the first time
        if len(df_pr) >= 4:
            # Normal WLS
            if np.all(x0 == 0):
                opt = scipy.optimize.least_squares(
                    pr_residuals, x0, jac_pr_residuals, args=(xsat_pr, pr, Wx))
                x0 = opt.x 
            # Robust WLS for position estimation
            opt = scipy.optimize.least_squares(
                 pr_residuals, x0, jac_pr_residuals, args=(xsat_pr, pr, Wx), loss='soft_l1')
            if opt.status < 1 or opt.status == 2:
                print(f'i = {i} position lsq status = {opt.status}')
            else:
                # Covariance estimation
                cov = np.linalg.inv(opt.jac.T @ Wx @ opt.jac)
                cov_x[i, :, :] = cov[:3, :3]
                x_wls[i, :] = opt.x[:3]
                x0 = opt.x
                 
        # Velocity estimation
        if len(df_prr) >= 4:
            if np.all(v0 == 0): # Normal WLS
                opt = scipy.optimize.least_squares(
                    prr_residuals, v0, jac_prr_residuals, args=(vsat, prr, x0, xsat_prr, Wv))
                v0 = opt.x
            # Robust WLS for velocity estimation
            opt = scipy.optimize.least_squares(
                prr_residuals, v0, jac_prr_residuals, args=(vsat, prr, x0, xsat_prr, Wv), loss='soft_l1')
            if opt.status < 1:
                print(f'i = {i} velocity lsq status = {opt.status}')
            else:
                # Covariance estimation
                cov = np.linalg.inv(opt.jac.T @ Wv @ opt.jac)
                cov_v[i, :, :] = cov[:3, :3]
                v_wls[i, :] = opt.x[:3]
                v0 = opt.x

    return utcTimeMillis, x_wls, v_wls, cov_x, cov_v

In [9]:
# Simple outlier detection and interpolation
def exclude_interpolate_outlier(x_wls, v_wls, cov_x, cov_v):
    # Up velocity / height threshold
    v_up_th = 2.6  # m/s  2.0 -> 2.6
    height_th = 200.0 # m
    v_out_sigma = 3.0 # m/s
    x_out_sigma = 30.0 # m
    
    # Coordinate conversion
    x_llh = np.array(pm.ecef2geodetic(x_wls[:, 0], x_wls[:, 1], x_wls[:, 2])).T
    x_llh_mean = np.nanmean(x_llh, axis=0)
    v_enu = np.array(pm.ecef2enuv(
        v_wls[:, 0], v_wls[:, 1], v_wls[:, 2], x_llh_mean[0], x_llh_mean[1])).T

    # Up velocity jump detection
    # Cars don't jump suddenly!
    idx_v_out = np.abs(v_enu[:, 2]) > v_up_th
    idx_v_out |= np.isnan(v_enu[:, 2])
    v_wls[idx_v_out, :] = np.nan
    cov_v[idx_v_out] = v_out_sigma**2 * np.eye(3)
    print(f'Number of velocity outliers {np.count_nonzero(idx_v_out)}')

    # Height check
    hmedian = np.nanmedian(x_llh[:, 2])
    idx_x_out = np.abs(x_llh[:, 2] - hmedian) > height_th
    idx_x_out |= np.isnan(x_llh[:, 2])
    x_wls[idx_x_out, :] = np.nan
    cov_x[idx_x_out] = x_out_sigma**2 * np.eye(3)
    print(f'Number of position outliers {np.count_nonzero(idx_x_out)}')

    # Interpolate NaNs at beginning and end of array
    x_df = pd.DataFrame({'x': x_wls[:, 0], 'y': x_wls[:, 1], 'z': x_wls[:, 2]})
    x_df = x_df.interpolate(limit_area='outside', limit_direction='both')

    # Interpolate all NaN data
    v_df = pd.DataFrame({'x': v_wls[:, 0], 'y': v_wls[:, 1], 'z': v_wls[:, 2]})
    v_df = v_df.interpolate(limit_area='outside', limit_direction='both')
    v_df = v_df.interpolate('spline', order=3)

    return x_df.to_numpy(), v_df.to_numpy(), cov_x, cov_v

In [10]:
# Kalman filter
def Kalman_filter(zs, us, cov_zs, cov_us, phone):
    # Parameters
    sigma_mahalanobis = 30.0  # Mahalanobis distance for rejecting innovation

    n, dim_x = zs.shape
    F = np.eye(3)  # Transition matrix
    H = np.eye(3)  # Measurement function

    # Initial state and covariance
    x = zs[0, :3].T  # State
    P = 5.0**2 * np.eye(3)  # State covariance
    I = np.eye(dim_x)

    x_kf = np.zeros([n, dim_x])
    P_kf = np.zeros([n, dim_x, dim_x])

    # Kalman filtering
    for i, (u, z) in enumerate(zip(us, zs)):
        # First step
        if i == 0:
            x_kf[i] = x.T
            P_kf[i] = P
            continue

        # Prediction step
        Q = cov_us[i] # Estimated WLS velocity covariance
        x = F @ x + u.T
        P = (F @ P) @ F.T + Q

        # Check outliers for observation
        d = distance.mahalanobis(z, H @ x, np.linalg.inv(P))

        # Update step
        if d < sigma_mahalanobis:
            R = cov_zs[i] # Estimated WLS position covariance
            y = z.T - H @ x
            S = (H @ P) @ H.T + R
            K = (P @ H.T) @ np.linalg.inv(S)
            x = x + K @ y
            P = (I - (K @ H)) @ P
        else:
            # If observation update is not available, increase covariance
            P += 10**2*Q

        x_kf[i] = x.T
        P_kf[i] = P

    return x_kf, P_kf


# Forward + backward Kalman filter and smoothing
def Kalman_smoothing(x_wls, v_wls, cov_x, cov_v, phone):
    n, dim_x = x_wls.shape

    # For some unknown reason, the speed estimation is wrong only for XiaomiMi8
    # so the variance is increased
    if phone == 'XiaomiMi8':
        v_wls = np.vstack([(v_wls[:-1, :] + v_wls[1:, :])/2, np.zeros([1, 3])])
        cov_v = 1000.0**2 * cov_v
         
    # Forward
    v = np.vstack([np.zeros([1, 3]), (v_wls[:-1, :] + v_wls[1:, :])/2])
    x_f, P_f = Kalman_filter(x_wls, v, cov_x, cov_v, phone)

    # Backward
    v = -np.flipud(v_wls)
    v = np.vstack([np.zeros([1, 3]), (v[:-1, :] + v[1:, :])/2])
    cov_xf = np.flip(cov_x, axis=0)
    cov_vf = np.flip(cov_v, axis=0)
    x_b, P_b = Kalman_filter(np.flipud(x_wls), v, cov_xf, cov_vf, phone)

    # Smoothing
    x_fb = np.zeros_like(x_f)
    P_fb = np.zeros_like(P_f)
    for (f, b) in zip(range(n), range(n-1, -1, -1)):
        P_fi = np.linalg.inv(P_f[f])
        P_bi = np.linalg.inv(P_b[b])

        P_fb[f] = np.linalg.inv(P_fi + P_bi)
        x_fb[f] = P_fb[f] @ (P_fi @ x_f[f] + P_bi @ x_b[b])

    return x_fb, x_f, np.flipud(x_b)

In [13]:
results = []

for i, (group_keys, trip_df) in enumerate(
        tqdm(gnss_df.groupby(['drive_id', 'device']))):

    drive, phone = group_keys
    tripID = f'{drive}/{phone}'
    print(tripID)

    # Point positioning
    utc, x_wls, v_wls, cov_x, cov_v = point_positioning(trip_df)

    # Remove velocity outliers
    x_wls, v_wls, cov_x, cov_v = exclude_interpolate_outlier(
        x_wls, v_wls, cov_x, cov_v
    )

    # Kalman smoothing
    x_kf, _, _ = Kalman_smoothing(x_wls, v_wls, cov_x, cov_v, phone)

    assert np.all(~np.isnan(x_kf))

    # build dataframe for this trip
    trip_result = pd.DataFrame({
        "utcTimeMillis": utc,
        "x_kf": x_kf[:, 0],
        "y_kf": x_kf[:, 1],
        "z_kf": x_kf[:, 2],
        "drive_id": drive,
        "device": phone
    })

    results.append(trip_result)

# combine all trips
kfgnss_df = pd.concat(results, ignore_index=True)

  0%|          | 0/170 [00:00<?, ?it/s]

2020-05-15-US-MTV-1/GooglePixel4XL


  0%|          | 0/3362 [00:00<?, ?it/s]

Number of velocity outliers 1
Number of position outliers 0
2020-05-21-US-MTV-1/GooglePixel4


  0%|          | 0/1952 [00:00<?, ?it/s]

...

Number of velocity outliers 0
Number of position outliers 0
2021-12-28-US-MTV-1/SamsungGalaxyS20Ultra


  0%|          | 0/1615 [00:00<?, ?it/s]

Number of velocity outliers 2
Number of position outliers 1
2021-12-28-US-MTV-1/XiaomiMi8


  0%|          | 0/1614 [00:00<?, ?it/s]

Number of velocity outliers 6
Number of position outliers 0


In [15]:
print(kfgnss_df)

        utcTimeMillis          x_kf          y_kf          z_kf  \
0       1589573679445 -2.693917e+06 -4.297456e+06  3.854196e+06   
1       1589573680445 -2.693917e+06 -4.297456e+06  3.854196e+06   
2       1589573681445 -2.693917e+06 -4.297456e+06  3.854196e+06   
3       1589573682445 -2.693917e+06 -4.297456e+06  3.854196e+06   
4       1589573683445 -2.693917e+06 -4.297456e+06  3.854196e+06   
...               ...           ...           ...           ...   
295628  1640724240000 -2.693854e+06 -4.297546e+06  3.854154e+06   
295629  1640724241000 -2.693854e+06 -4.297546e+06  3.854154e+06   
295630  1640724242000 -2.693854e+06 -4.297546e+06  3.854154e+06   
295631  1640724243000 -2.693854e+06 -4.297546e+06  3.854154e+06   
295632  1640724244000 -2.693854e+06 -4.297546e+06  3.854154e+06   

                   drive_id          device  
0       2020-05-15-US-MTV-1  GooglePixel4XL  
1       2020-05-15-US-MTV-1  GooglePixel4XL  
2       2020-05-15-US-MTV-1  GooglePixel4XL  
3       202

In [20]:
kfgnss = kfgnss_df.merge(
    gnss_df,
    on=["drive_id","device","utcTimeMillis"],
    how="left"
)

In [23]:
# shrink large gnss file
gnsskf_df = kfgnss[['utcTimeMillis', 'device', 'drive_id', 'Svid', 'Cn0DbHz', 'MultipathIndicator', 'SvElevationDegrees', 'x_kf', 'y_kf', 'z_kf']]

In [24]:
%store gnsskf_df

Stored 'gnsskf_df' (DataFrame)


In [11]:
# Point positioning
utc, drive_id, device, x_wls, v_wls, cov_x, cov_v = point_positioning(gnss_df)

  0%|          | 0/260779 [00:00<?, ?it/s]

i = 5065 velocity lsq status = 0
i = 5107 position lsq status = 0
i = 5111 position lsq status = 0
i = 5119 position lsq status = 0
i = 5276 position lsq status = 0
i = 5468 position lsq status = 0
i = 5478 position lsq status = 0
i = 5512 position lsq status = 0
i = 5514 position lsq status = 0
i = 5599 velocity lsq status = 0
i = 5877 position lsq status = 0
i = 5888 position lsq status = 0
i = 5891 position lsq status = 0
i = 5915 position lsq status = 0
i = 5990 position lsq status = 0
i = 6004 position lsq status = 0
i = 6098 position lsq status = 0
i = 6124 position lsq status = 0
i = 6126 position lsq status = 0
i = 6167 position lsq status = 0
i = 6187 position lsq status = 0
i = 6206 position lsq status = 0
i = 6209 position lsq status = 0
i = 6213 position lsq status = 0
i = 6217 position lsq status = 0
i = 6219 position lsq status = 0
i = 6231 position lsq status = 0
i = 6235 position lsq status = 0
i = 6239 position lsq status = 0
i = 6451 position lsq status = 0
i = 6569 p

NameError: name 'pm' is not defined

In [15]:
# Exclude velocity outliers
x_wls, v_wls, cov_x, cov_v = exclude_interpolate_outlier(x_wls, v_wls, cov_x, cov_v)

Number of velocity outliers 8186
Number of position outliers 40349


NameError: name 'phone' is not defined

In [17]:
# Kalman smoothing
x_kf, _, _ = Kalman_smoothing(x_wls, v_wls, cov_x, cov_v, phone)

In [ ]:
# gnss_df = carrier_smoothing(gnss_df)

In [ ]:
# gnss_list = []

# for file in gnss_files:
#     gnss_file = pd.read_csv(file)
#     parts = file.split(os.sep) # eg ['train', '2021-12-28-US-MTV-1', 'GooglePixel5', 'device_imu.csv']
#     gnss_file["drive_id"] = parts[-3] 
#     gnss_file["device"] = parts[-2]
#     gnss_list.append(gnss_file)

# # combine all
# gnss_df = pd.concat(gnss_list, ignore_index=True)
# print(gnss_df.head())
# print()
# print(gnss_df["device"].unique())
# print()
# print(gnss_df["drive_id"].unique())

# shrink large gnss file
# gnss_df = gnss_df[['utcTimeMillis', 'device', 'drive_id', 'Svid', 'Cn0DbHz', 'MultipathIndicator', 'SvElevationDegrees', 'WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters']]

In [ ]:
# %store gnss_df

In [23]:
trip_kf_df = pd.DataFrame(x_kf, columns=['x_kf', 'y_kf', 'z_kf'])
trip_kf_df['utcTimeMillis'] = utc

print(trip_kf_df)

                x_kf          y_kf          z_kf  utcTimeMillis
0      -2.693907e+06 -4.297449e+06  3.854207e+06  1609800004433
1      -2.693914e+06 -4.297456e+06  3.854198e+06  1609800005433
2      -2.693915e+06 -4.297456e+06  3.854195e+06  1609800006433
3      -2.693916e+06 -4.297455e+06  3.854194e+06  1609800007433
4      -2.693916e+06 -4.297455e+06  3.854194e+06  1609800008433
...              ...           ...           ...            ...
260774 -2.693853e+06 -4.297546e+06  3.854153e+06  1596756049433
260775 -2.693853e+06 -4.297546e+06  3.854152e+06  1596756050433
260776 -2.693853e+06 -4.297546e+06  3.854152e+06  1596756051433
260777 -2.693853e+06 -4.297546e+06  3.854151e+06  1596756052433
260778 -2.693854e+06 -4.297546e+06  3.854151e+06  1596756053433

[260779 rows x 4 columns]
